# Load a Shout Measurement File from an Experiment on POWDER





In [ ]:
# import os

# user = "npatwari"
# repo = "QPSK-RX-Tutorial"

# # remove local directory if it already exists
# if os.path.isdir(repo):
#     !rm -rf {repo}

# !git clone https://github.com/{user}/{repo}.git

General info about Colab from Google:

> To execute the code in any cell, select it with a click and then either press the play button to the left of the code, or use the keyboard shortcut "Command/Ctrl+Enter". To edit the code, just click the cell and start editing.


### Import Libraries

In [1]:
import os
import json
import math
import numpy as np
from scipy import signal
import h5py
import matplotlib.pyplot as plt
from matplotlib import rc
import datetime
import scipy.signal as signal
rc('xtick', labelsize=14)
rc('ytick', labelsize=14)

## Load recorded data into the environment

Here we define some functions that load the measured data and the Shout metadata about the recording from the hdf5 file.

In [3]:
def get_time_string(timestamp):
    '''
    Helper function to get data and time from timestamp
    INPUT: timestamp
    OUTPUT: data and time. Example: 01-04-2023, 19:50:27
    '''
    date_time = datetime.datetime.fromtimestamp(int(timestamp))
    return date_time.strftime("%m-%d-%Y, %H:%M:%S")

def JsonLoad(folder, json_file):
    '''
    Load parameters from the saved json file
    INPUT
    ----
        folder: path to the measurement folder. Example: "SHOUT/Results/Shout_meas_01-04-2023_18-50-26"
        json_file: the json file with all the specifications. Example: '/save_iq_w_tx_gold.json'
    OUTPUT
    ----
        samps_per_chip: samples per chip
        wotxrepeat: number of repeating IQ sample collection w/o transmission. Used as an input to
        traverse_dataset() func
        rxrate: sampling rate at the receiver side
    '''
    config_file = folder+'/'+json_file
    config_dict = json.load(open(config_file))[0]
    nsamps = config_dict['nsamps']
    rxrate = config_dict['rxrate']
    rxfreq = config_dict['rxfreq']
    wotxrepeat = config_dict['wotxrepeat']
    rxrepeat = config_dict['rxrepeat']
    txnodes = config_dict['txclients']
    rxnodes = config_dict['rxclients']

    return rxrepeat, rxrate, txnodes, rxnodes

def traverse_dataset(meas_folder):
    '''
    Load data from hdf5 format measurement file
    INPUT
    ----
        meas_folder: path to the measurement folder. Example: "SHOUT/Results/Shout_meas_01-04-2023_18-50-26"
    OUTPUT
    ----
        data: Collected IQ samples w/ transmission. It is indexed by the transmitter name
        noise: Collected IQ samples w/o transmission. It is indexed by the transmitter name
        txrxloc: transmitter and receiver names
    '''
    data = {}
    noise = {}
    txrxloc = {}

    dataset = h5py.File(meas_folder + '/measurements.hdf5', "r") #meas_folder
    print("Dataset meta data:", list(dataset.attrs.items()))
    for cmd in dataset.keys():
        print("Command:", cmd)
        if cmd == 'saveiq':
            cmd_time = list(dataset[cmd].keys())[0]
            print("  Timestamp:", get_time_string(cmd_time))
            print("  Command meta data:", list(dataset[cmd][cmd_time].attrs.items()))
            for rx_gain in dataset[cmd][cmd_time].keys():
                print("   RX gain:", rx_gain)
                for rx in dataset[cmd][cmd_time][rx_gain].keys():
                    print("     RX:", rx)
                    print("       Measurement items:", list(dataset[cmd][cmd_time][rx_gain][rx].keys()))
        elif cmd == 'saveiq_w_tx':
            cmd_time = list(dataset[cmd].keys())[0]
            print("  Timestamp:", get_time_string(cmd_time))
            print("  Command meta data:", list(dataset[cmd][cmd_time].attrs.items()))
            for tx in dataset[cmd][cmd_time].keys():
                print("   TX:", tx)

                if tx == 'wo_tx':
                    for rx_gain in dataset[cmd][cmd_time][tx].keys():
                        print("       RX gain:", rx_gain)
                        #print(dataset[cmd][cmd_time][tx][rx_gain].keys())
                        for rx in dataset[cmd][cmd_time][tx][rx_gain].keys():
                            print("         RX:", rx)
                            #print("           Measurement items:", list(dataset[cmd][cmd_time][tx][rx_gain][rx].keys()))
                            repeat = np.shape(dataset[cmd][cmd_time][tx][rx_gain][rx]['rxsamples'])[0]
                            print("         repeat", repeat)

                            samplesNotx =  dataset[cmd][cmd_time][tx][rx_gain][rx]['rxsamples'][:repeat, :]
                            namelist = rx.split('-')
                            noise[namelist[1]] = samplesNotx
                else:
                    for tx_gain in dataset[cmd][cmd_time][tx].keys():
                        print("     TX gain:", tx_gain)
                        for rx_gain in dataset[cmd][cmd_time][tx][tx_gain].keys():
                            print("       RX gain:", rx_gain)
                            #print(dataset[cmd][cmd_time][tx][tx_gain][rx_gain].keys())
                            for rx in dataset[cmd][cmd_time][tx][tx_gain][rx_gain].keys():
                                repeat = np.shape(dataset[cmd][cmd_time][tx][tx_gain][rx_gain][rx]['rxsamples'])[0]
                                print("         RX:", rx, "; samples shape", np.shape(dataset[cmd][cmd_time][tx][tx_gain][rx_gain][rx]['rxsamples']))
                                #print("         Measurement items:", list(dataset[cmd][cmd_time][tx][tx_gain][rx_gain][rx].keys()))
                                # print("         rxloc", (dataset[cmd][cmd_time][tx][tx_gain][rx_gain][rx]['rxloc'][0]))
                                # peak avg check
                                txrxloc.setdefault(tx, []).extend([rx]*repeat)
                                rxsamples = dataset[cmd][cmd_time][tx][tx_gain][rx_gain][rx]['rxsamples'][:repeat, :]
                                data.setdefault(tx, []).append(np.array(rxsamples))
        else:
            print('Unsupported command: ', cmd)

    return data, noise, txrxloc


This code loads the IQ data from a file in this folder named measurements.hdf5 and prints info about the measurements in it.  The primary data is in the first return `rx_data`, which is a python dictionary that stores all of the complex-valued measurements, indexed by the transmitter and receiver names, and repetition number.

In [4]:
# Load parameters from the JSON file which describe what was measured
ShoutFolderName = "Shout_meas_03-12-2026_21-27-21"

# remove local directory if it already exists
if os.path.isdir(ShoutFolderName):
    !rm -rf {ShoutFolderName}

!unzip -q {ShoutFolderName}.zip

✋ Look in the file directory (folder icon left of the colab editor) to find the file name of the JSON file in the Shout measurement folder, and put it in for `jsonfile` here.

In [6]:
folder = "./" + ShoutFolderName
print(folder)
jsonfile = "save_iq_w_tx_file.json"  # For you to enter
rxrepeat, samp_rate, txlocs, rxlocs = JsonLoad(folder, jsonfile)
# Load data from the HDF5 file, save IQ sample arrays
rx_data, noise_v, txrxloc = traverse_dataset(folder)
print(txlocs)

./Shout_meas_03-12-2026_21-27-21
Dataset meta data: [('shout_version', '495923e')]
Command: saveiq_w_tx
  Timestamp: 03-13-2026, 03:27:22
  Command meta data: [('cmd', 'save_iq_w_tx'), ('nsamps', np.int64(8192)), ('rxfreq', np.float64(3395375000.0)), ('rxgain', np.float64(30.0)), ('rxrate', np.float64(250000.0)), ('rxrepeat', np.int64(4)), ('rxwait_max', np.int64(2000)), ('rxwait_min', np.int64(50)), ('rxwait_random', np.True_), ('rxwait_res', 'ms'), ('start_time', np.float64(1773372448.0)), ('sync', np.True_), ('timeout', np.int64(30)), ('timezone', 'US/Mountain'), ('txfile', '/local/repository/shout/signal_library/channel-sounding-OFDM-packet-10-03-26.iq'), ('txfreq', np.float64(3395375000.0)), ('txgain', np.float64(27.0)), ('txrate', np.float64(250000.0)), ('txwait', np.int64(3)), ('use_lo_offset', np.True_), ('wotxrepeat', np.int64(1))]
   TX: 0
     TX gain: 27.0
       RX gain: 30.0
         RX: cbrssdr1-bes-comp ; samples shape (4, 8192)
         RX: cbrssdr1-browning-comp ; sam

In [7]:
print(txrxloc)

{'0': ['cbrssdr1-bes-comp', 'cbrssdr1-bes-comp', 'cbrssdr1-bes-comp', 'cbrssdr1-bes-comp', 'cbrssdr1-browning-comp', 'cbrssdr1-browning-comp', 'cbrssdr1-browning-comp', 'cbrssdr1-browning-comp', 'cbrssdr1-fm-comp', 'cbrssdr1-fm-comp', 'cbrssdr1-fm-comp', 'cbrssdr1-fm-comp'], '0+cbrssdr1-bes-comp': ['cbrssdr1-browning-comp', 'cbrssdr1-browning-comp', 'cbrssdr1-browning-comp', 'cbrssdr1-browning-comp', 'cbrssdr1-fm-comp', 'cbrssdr1-fm-comp', 'cbrssdr1-fm-comp', 'cbrssdr1-fm-comp'], '0+cbrssdr1-bes-comp+cbrssdr1-browning-comp': ['cbrssdr1-fm-comp', 'cbrssdr1-fm-comp', 'cbrssdr1-fm-comp', 'cbrssdr1-fm-comp'], '0+cbrssdr1-bes-comp+cbrssdr1-fm-comp': ['cbrssdr1-browning-comp', 'cbrssdr1-browning-comp', 'cbrssdr1-browning-comp', 'cbrssdr1-browning-comp'], '0+cbrssdr1-browning-comp': ['cbrssdr1-bes-comp', 'cbrssdr1-bes-comp', 'cbrssdr1-bes-comp', 'cbrssdr1-bes-comp', 'cbrssdr1-fm-comp', 'cbrssdr1-fm-comp', 'cbrssdr1-fm-comp', 'cbrssdr1-fm-comp'], '0+cbrssdr1-browning-comp+cbrssdr1-fm-comp': ['

In [8]:
rx_data

{'0': [array([[ 3.051851e-05+0.000000e+00j,  0.000000e+00-3.051851e-05j,
          -3.051851e-05-3.051851e-05j, ...,  3.051851e-05+3.051851e-05j,
           0.000000e+00+6.103702e-05j,  3.051851e-05+3.051851e-05j],
         [ 6.103702e-05+0.000000e+00j, -6.103702e-05+3.051851e-05j,
          -3.051851e-05+3.051851e-05j, ...,  0.000000e+00-3.051851e-05j,
          -3.051851e-05-3.051851e-05j,  6.103702e-05+3.051851e-05j],
         [ 9.155553e-05+6.103702e-05j,  0.000000e+00+6.103702e-05j,
           3.051851e-05+0.000000e+00j, ..., -3.051851e-05-6.103702e-05j,
           3.051851e-05+0.000000e+00j, -3.051851e-05-6.103702e-05j],
         [ 9.155553e-05-6.103702e-05j,  3.051851e-05-6.103702e-05j,
          -9.155553e-05-6.103702e-05j, ...,  0.000000e+00-6.103702e-05j,
           0.000000e+00+0.000000e+00j,  0.000000e+00-6.103702e-05j]],
        dtype=complex64),
  array([[-3.051851e-05-3.051851e-05j,  0.000000e+00-3.051851e-05j,
           3.051851e-05-6.103702e-05j, ...,  0.000000e+00+0.

In [10]:
txrxloc.keys()

dict_keys(['0', '0+cbrssdr1-bes-comp', '0+cbrssdr1-bes-comp+cbrssdr1-browning-comp', '0+cbrssdr1-bes-comp+cbrssdr1-fm-comp', '0+cbrssdr1-browning-comp', '0+cbrssdr1-browning-comp+cbrssdr1-fm-comp', '0+cbrssdr1-fm-comp'])

In [24]:
for tx in txrxloc.keys():
    print(tx, ": ", txrxloc[tx])
    print(rx_data[tx])

0 :  ['cbrssdr1-bes-comp', 'cbrssdr1-bes-comp', 'cbrssdr1-bes-comp', 'cbrssdr1-bes-comp', 'cbrssdr1-browning-comp', 'cbrssdr1-browning-comp', 'cbrssdr1-browning-comp', 'cbrssdr1-browning-comp', 'cbrssdr1-fm-comp', 'cbrssdr1-fm-comp', 'cbrssdr1-fm-comp', 'cbrssdr1-fm-comp']
[array([[ 3.051851e-05+0.000000e+00j,  0.000000e+00-3.051851e-05j,
        -3.051851e-05-3.051851e-05j, ...,  3.051851e-05+3.051851e-05j,
         0.000000e+00+6.103702e-05j,  3.051851e-05+3.051851e-05j],
       [ 6.103702e-05+0.000000e+00j, -6.103702e-05+3.051851e-05j,
        -3.051851e-05+3.051851e-05j, ...,  0.000000e+00-3.051851e-05j,
        -3.051851e-05-3.051851e-05j,  6.103702e-05+3.051851e-05j],
       [ 9.155553e-05+6.103702e-05j,  0.000000e+00+6.103702e-05j,
         3.051851e-05+0.000000e+00j, ..., -3.051851e-05-6.103702e-05j,
         3.051851e-05+0.000000e+00j, -3.051851e-05-6.103702e-05j],
       [ 9.155553e-05-6.103702e-05j,  3.051851e-05-6.103702e-05j,
        -9.155553e-05-6.103702e-05j, ...,  0.00

In [25]:
print(list(txrxloc.keys()))

['0', '0+cbrssdr1-bes-comp', '0+cbrssdr1-bes-comp+cbrssdr1-browning-comp', '0+cbrssdr1-bes-comp+cbrssdr1-fm-comp', '0+cbrssdr1-browning-comp', '0+cbrssdr1-browning-comp+cbrssdr1-fm-comp', '0+cbrssdr1-fm-comp']
